Policy Iteration Algorithm : 
Implémenter l'algorithme de Policy Iteration pour résoudre le problème du Frozen Lake et puis afficher les Q-table et Q-policy obtenues.

In [19]:
%run "Creation de Grille.ipynb"

Exception: File `'Creation de Grille.ipynb'` not found.

In [14]:
def policy_evaluation(policy, n, m, valid_actions, terminal, reward):
    S = n*m
    V = np.zeros(S)

    while True:
        delta = 0.0
        for s in range(S):
            if terminal[s]:
                continue
            a = policy[s]
            s2 = step_det(n, m, s, a)
            v_new = reward[s2] + (0.0 if terminal[s2] else gamma * V[s2])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        if delta < theta:
            break

    return V


In [15]:
def compute_Q_from_V(V, n, m, valid_actions, terminal, reward, gamma=0.3):
    S = n*m
    Q = np.full((S, 4), -1e9)  # actions invalides = très faible

    for s in range(S):
        if terminal[s]:
            continue
        for a in valid_actions[s]:
            s2 = step_det(n, m, s, a)
            Q[s, a] = reward[s2] + (0.0 if terminal[s2] else gamma * V[s2])

    return Q


In [16]:
def improve_policy(Q, valid_actions, terminal):
    S = Q.shape[0]
    new_policy = np.zeros(S, dtype=int)

    for s in range(S):
        if terminal[s]:
            new_policy[s] = 0
        else:
            new_policy[s] = max(valid_actions[s], key=lambda a: Q[s, a])

    return new_policy


In [17]:


def policy_iteration(n, m, valid_actions, terminal, reward, gamma=0.2, theta=1e-6, max_iter=1000):
    S = n*m

    # policy initiale random (valide)
    policy = np.zeros(S, dtype=int)
    for s in range(S):
        if terminal[s]:
            policy[s] = 0
        else:
            policy[s] = random.choice(valid_actions[s])

    for it in range(max_iter):
        V = policy_evaluation(policy, n, m, valid_actions, terminal, reward)
        Q = compute_Q_from_V(V, n, m, valid_actions, terminal, reward, gamma)
        new_policy = improve_policy(Q, valid_actions, terminal)

        if np.array_equal(new_policy, policy):
            print("✅ Convergence atteinte en", it+1, "itérations")
            return policy, V, Q

        policy = new_policy

    print("⚠️ Max itérations atteintes sans convergence")
    return policy, V, Q


In [18]:
policy, V, Q = policy_iteration(n, m, valid_actions, terminal, reward, gamma=0.2, theta=1e-6)

print("\n--- Q-table (arrondi) ---")
print(np.round(Q, 3))

print("\n--- Q-policy (vecteur d'actions) ---")
print(policy)


NameError: name 'n' is not defined

In [ ]:
ARROWS = {0:"↑", 1:"→", 2:"↓", 3:"←"}

def plot_policy_grid_clean(
    n, m, policy, holes, goal_state, start_state,
    title="Politique finale"
):
    holes_set = set(holes)

    # 0: normal, 1: hole, 2: goal, 3: start
    cat = np.zeros((n, m), dtype=int)

    for s in holes_set:
        r, c = divmod(s, m)
        cat[r, c] = 1

    rg, cg = divmod(goal_state, m)
    cat[rg, cg] = 2

    rs, cs = divmod(start_state, m)
    cat[rs, cs] = 3

    cmap = ListedColormap(["white", "deepskyblue", "lightgreen", "orange"])

    plt.figure(figsize=(7, 7))
    plt.imshow(cat, cmap=cmap, origin="upper")

    # Grille sans numéros
    plt.xticks(np.arange(-0.5, m, 1), [])
    plt.yticks(np.arange(-0.5, n, 1), [])
    plt.grid(True, which="both", color="black", linewidth=1)

    for r in range(n):
        for c in range(m):
            s = r*m + c

            # Goal
            if s == goal_state:
                plt.text(c, r, "G", ha="center", va="center",
                         fontsize=18, weight="bold")
                continue

            # Hole
            if s in holes_set:
                plt.text(c, r, "H", ha="center", va="center",
                         fontsize=18, weight="bold")
                continue

            # Start
            if s == start_state:
                plt.text(c, r, "S", ha="center", va="center",
                         fontsize=18, weight="bold")
                continue

            # Flèche uniquement sur états normaux
            a = int(policy[s])
            plt.text(c, r, ARROWS[a], ha="center", va="center",
                     fontsize=22, weight="bold")
    plt.title(title)
    plt.show()
plot_policy_grid_clean(
n=7,
m=7,
policy=policy,
holes=holes,
goal_state=goal_state,
start_state=start_state,
title="Policy Iteration – Politique finale"
)


NameError: name 'policy' is not defined